In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    LSTM,
    Dense,
    Dropout,
    Bidirectional
)

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [3]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

file_path = os.path.join(BASE_DIR, "data", "processed", "cleaned.csv")

df = pd.read_csv(file_path)

In [4]:
X = df["clean_text"].fillna("").astype(str)
y = df["sentiment"].astype(int)

print(y.value_counts())

sentiment
0    29000
1    28994
Name: count, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(46395,)
(11599,)


In [6]:
max_words = 20000
max_len = 300
# max_len = 150

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
# tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

print(X_train_pad.shape)

(46395, 300)


In [7]:
word_index = tokenizer.word_index
# vocab_size = len(word_index) + 1
vocab_size = max_words

print("Vocabulary size:", vocab_size)

Vocabulary size: 20000


In [8]:
embedding_index = {}

glove_path = os.path.join(
    BASE_DIR,
    "embeddings",
    "glove.6B.100d.txt"
)

with open(glove_path, encoding="utf8") as f:
    for line in f:
        values = line.split()

        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')

        embedding_index[word] = vector

print("Loaded word vectors:", len(embedding_index))

Loaded word vectors: 400000


In [9]:
embedding_dim = 100
max_words = 20000

embedding_matrix = np.zeros((max_words, embedding_dim))

for word, i in word_index.items():

    if i >= max_words:
        continue

    embedding_vector = embedding_index.get(word)

    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print("Embedding matrix shape:",
      embedding_matrix.shape)

print("Nonzero values:",
      np.count_nonzero(embedding_matrix))

# debugging check
print(embedding_matrix[100][:10])
print(np.sum(embedding_matrix[100]))

Embedding matrix shape: (20000, 100)
Nonzero values: 1434100
[ 0.41711    -0.10176     0.058147   -0.18332    -0.44457999 -0.17851999
 -0.34391999 -0.077147    0.58521003 -0.52752   ]
6.792773924069479


Model

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Bidirectional,
    GRU,
    LSTM,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

model_hbgru_lstm = Sequential([

    # GloVe Embedding Layer
    Embedding(
        input_dim=max_words,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=False
    ),

    # Bidirectional GRU
    Bidirectional(
        GRU(
            128,
            return_sequences=True,
            dropout=0.2,
            recurrent_dropout=0.2
        )
    ),

    # LSTM Layer
    LSTM(
        128,
        dropout=0.2,
        recurrent_dropout=0.2
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        1,
        activation='sigmoid'
    )
])

model_hbgru_lstm.summary()

c:\Users\USER\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,000,000 (7.63 MB)

In [11]:
model_hbgru_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [12]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

In [13]:
history_hbgru_lstm = model_hbgru_lstm.fit(
    X_train_pad,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2672s 5s/step - accuracy: 0.4981 - loss: 0.6939 - val_accuracy: 0.5021 - val_loss: 0.6931
Epoch 2/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2694s 5s/step - accuracy: 0.5052 - loss: 0.6932 - val_accuracy: 0.4995 - val_loss: 0.6934
Epoch 3/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2617s 5s/step - accuracy: 0.5017 - loss: 0.6932 - val_accuracy: 0.4991 - val_loss: 0.6931
Epoch 4/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2526s 4s/step - accuracy: 0.4998 - loss: 0.6932 - val_accuracy: 0.5021 - val_loss: 0.6932
Epoch 5/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2476s 4s/step - accuracy: 0.4983 - loss: 0.6931 - val_accuracy: 0.4989 - val_loss: 0.6930
Epoch 6/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2481s 4s/step - accuracy: 0.5019 - loss: 0.6929 - val_accuracy: 0.4984 - val_loss: 0.6930
Epoch 7/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2488s 4s/step - accuracy: 0.5012 - loss: 0.6927 - val_accuracy: 0.4981 - val_loss: 0.6931
Epoch 8/20
580/580 ━━━━━━━━━━━━━━━━━━━━ 2498s 4s/step - accuracy: 0.4998 - loss: 0.6926 - 

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report
)

y_prob = model_hbgru_lstm.predict(X_test_pad)

y_pred = (y_prob > 0.5).astype(int)

print("Accuracy:",
      accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

363/363 ━━━━━━━━━━━━━━━━━━━━ 71s 191ms/step
Accuracy: 0.4998706785067678
              precision    recall  f1-score   support

           0       0.49      0.00      0.01      5800
           1       0.50      1.00      0.67      5799

    accuracy                           0.50     11599
   macro avg       0.49      0.50      0.34     11599
weighted avg       0.49      0.50      0.34     11599

